<a href="https://colab.research.google.com/github/hfelizzola/Investigaciones-de-Operaciones-I/blob/main/programacion-entera/programacion_entera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación Entera Mixta

In [ ]:
!pip install gamspy --quiet

In [ ]:
# Importar librerias
import numpy as np # Manejar vectores y matrices
import pandas as pd # Manejar tablas
import matplotlib.pyplot as plt # Generar gráficos
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sense, Sum

## Ejemplo 1: Capital Budgeting IP (Stockco)
Stockco considera cuatro inversiones. Los datos de cada inversión se presentan en la Tabla. Stockco tiene un presupuesto total de $\$14.000$ disponible para el Año 1, y $\$10.000$ para el Año 2. El objetivo es maximizar el VPN total.

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-c3ow{border-color:inherit;text-align:center;vertical-align:top}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0pky"></th>
    <th class="tg-c3ow">Inv. 1</th>
    <th class="tg-c3ow">Inv. 2</th>
    <th class="tg-c3ow">Inv. 3</th>
    <th class="tg-c3ow">Inv. 4</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0pky">VPN (Valor Presente Neto)</td>
    <td class="tg-c3ow">16</td>
    <td class="tg-c3ow">22</td>
    <td class="tg-c3ow">12</td>
    <td class="tg-c3ow">8</td>
  </tr>
  <tr>
    <td class="tg-0pky">Costo Año 1</td>
    <td class="tg-c3ow">5</td>
    <td class="tg-c3ow">7</td>
    <td class="tg-c3ow">4</td>
    <td class="tg-c3ow">3</td>
  </tr>
  <tr>
    <td class="tg-0pky">Costo Año 2</td>
    <td class="tg-c3ow">2</td>
    <td class="tg-c3ow">3</td>
    <td class="tg-c3ow">1</td>
    <td class="tg-c3ow">4</td>
  </tr>
</tbody>
</table>

Se define una variable binaria $x_j$ para cada decisión de inversión:
$$
x_j = \begin{cases}
1 & \text{si la inversión } j \text{ se realiza} \\
0 & \text{en caso contrario}
\end{cases} \quad \text{para } j=1, 2, 3, 4
$$

**Función Objetivo:**
Se busca maximizar el VPN total:
$$
\max z = 16x_1 + 22x_2 + 12x_3 + 8x_4
$$

**Restricciones:**
Ahora hay dos restricciones presupuestarias, una para cada año:
\begin{align*}
5x_1 + 7x_2 + 4x_3 + 3x_4 & \le 14 \quad \text{(Presupuesto Año 1)} \\
2x_1 + 3x_2 + 1x_3 + 4x_4 & \le 10 \quad \text{(Presupuesto Año 2)}
\end{align*}

se pueden añadir las siguientes restricciones:

**1. Se puede invertir en como máximo tres inversiones:**
$$
x_1 + x_2 + x_3 + x_4 \le 3
$$

**2. Se invierte en la inversión 2, si se invierte en la 1:**
$$
x_2 \le x_1 \quad (\text{o } x_2 - x_1 \le 0)
$$

**3. Si Stockco invierte en la inversión 2, no puede invertir en la 4:**
$$
x_2 + x_4 \le 1
$$

## Modelo Matemático

In [ ]:
# Paso 1: Crear contenedor del modelo
m = Container()

# Paso 2: Definir conjuntos
inv = Set(m, name="inv", records=["inv1", "inv2", "inv3", "inv4"])
anno = Set(m, name="anno", records=["año1", "año2"])

# Paso 3: Parámetros del modelo

# Valor presente neto (VPN) de cada inversión
vpn = Parameter(m, name="vpn", domain=[inv], records=np.array([16,22,12,8]))

# Capital disponible por año
cap_disp = Parameter(m, name="cap_disp", domain=[anno], records=np.array([14,10]))

# Capital requerido por cada inverión en cada año
cap_req = Parameter(m, name="cap_req", domain=[anno, inv], records=np.array([
   [5,7,4,3],
   [2,3,1,4],
]))

# Paso 6: Variables de decision
# Indica si se invierte en i
x = Variable(m, name="x", type="binary", domain=[inv])

# Paso 5: Definir restricciones

# Restricción de capacidad por departamento
res_cap_req = Equation(m, name="res_cap_req", domain=[anno])
res_cap_req[...] = Sum(inv, cap_req[anno, inv] * x[inv]) <= cap_disp[anno]

# Restricciones de máximo
max_inv = Equation(m, name="max_inv")
max_inv[...] = Sum(inv, x[inv]) <= 3

# Se invierte en la inversión 2, si se invierte en la inversión 1
inv_2 = Equation(m, name="inv_2")
inv_2[...] = x["inv2"] <= x["inv1"]

# Las inversiones 2 y 4 son excluentes
inv_4 = Equation(m, name="inv_4")
inv_4[...] = x["inv2"] + x["inv4"] <= 1

# Paso 6: Función objetivo -> maximizar el valor presente neto de las inversiones
objetivo = Sum(inv, vpn[inv] * x[inv])

# Paso 7: Crear y resolver el modelo
modelo = Model(m,
               name="inversion",
               equations=[res_cap_req, max_inv, inv_2, inv_4],
               sense=Sense.MAX,
               objective=objetivo,
               problem="MIP")
modelo.solve()


,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,38.0,6,5,MIP,CPLEX,0.002


In [ ]:
x.records

,inv,level,marginal,lower,upper,scale
0,inv1,1.0,16.0,0.0,1.0,1.0
1,inv2,1.0,22.0,0.0,1.0,1.0
2,inv3,0.0,12.0,0.0,1.0,1.0
3,inv4,0.0,8.0,0.0,1.0,1.0


## Ejemplo 3: Gandhi Cloth Company

Gandhi fabrica 3 tipos de ropa (camisas, shorts, pantalones). Para fabricar cualquier cantidad de un tipo de ropa, se debe rentar maquinaria. Los costos fijos de renta son: Camisas $\$200$ semana, Shorts $\$150$ semana, Pantalones $\$100$ semana.

Los recursos disponibles son 150 horas de labor y 160 sq yd de tela. Los datos de producción y costos se muestran en las Tablas. El objetivo es maximizar la ganancia semanal.

<style type="text/css">
.tg  {border-collapse:collapse;border-color:#ccc;border-spacing:0;border-style:solid;border-width:1px;}
.tg td{background-color:#fff;border-color:#ccc;border-style:solid;border-width:0px;color:#333;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#f0f0f0;border-color:#ccc;border-style:solid;border-width:0px;color:#333;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-c3ow{border-color:inherit;text-align:center;vertical-align:top}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0pky">Tipo de Ropa</th>
    <th class="tg-c3ow">Labor (Horas)</th>
    <th class="tg-c3ow">Tela (Square Yards)</th>
    <th class="tg-0pky">Costo Unitario (\$)</th>
    <th class="tg-0pky">Precio (\$)</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0pky">Camisa</td>
    <td class="tg-c3ow">3</td>
    <td class="tg-c3ow">4</td>
    <td class="tg-0pky">6</td>
    <td class="tg-0pky">12</td>
  </tr>
  <tr>
    <td class="tg-0pky">Shorts</td>
    <td class="tg-c3ow">2</td>
    <td class="tg-c3ow">3</td>
    <td class="tg-0pky">4</td>
    <td class="tg-0pky">8</td>
  </tr>
  <tr>
    <td class="tg-0pky">Pantalones</td>
    <td class="tg-c3ow">6</td>
    <td class="tg-c3ow">4</td>
    <td class="tg-0pky">8</td>
    <td class="tg-0pky">15</td>
  </tr>
</tbody>
</table>

**Definición de Variables:**

Se necesitan dos tipos de variables:

1. Variables de producción (enteras):
* $x_1 = \text{número de camisas producidas}$,
* $x_2 = \text{número de shorts}$
* $x_3 = \text{número de pantalones}$.

2. Variables binarias de costo fijo

$$
y_j = \begin{cases}
    1 & \text{si se produce cualquier cantidad de ropa } j \ (x_j > 0) \\
    0 & \text{si no se produce ropa } j \ (x_j = 0)
    \end{cases}
$$

**Función Objetivo:**

La ganancia es (Ingresos) - (Costos Variables) - (Costos Fijos de Renta).

La ganancia por unidad es:
* Camisa = $12-6=6$,
* Shorts = $8-4=4$,
* Pantalones = $15-8=7$.

Por tanto la utilidad total es:

$$
\max z = (6x_1 + 4x_2 + 7x_3) - (200y_1 + 150y_2 + 100y_3)
$$

**Restricciones:**

\begin{align*}
3x_1 + 2x_2 + 6x_3 & \le 150 \quad \text{(Restricción de Labor)} \\
4x_1 + 3x_2 + 4x_3 & \le 160 \quad \text{(Restricción de Tela)}
\end{align*}


Necesitamos vincular $x_j$ con $y_j$. Forzamos a $y_j=1$ si $x_j > 0$ usando una constante $M_j$ (un número grande que representa la producción máxima posible).

\begin{align*}
x_1 & \le M_1 y_1 \quad \text{(ej. } M_1 = 40 \text{ camisas)} \\
x_2 & \le M_2 y_2 \quad \text{(ej. } M_2 = 53 \text{ shorts)} \\
x_3 & \le M_3 y_3 \quad \text{(ej. } M_3 = 25 \text{ pantalones)}
\end{align*}

Y las variables $x_j$ deben ser enteras $\ge 0$, y $y_j$ deben ser binarias.


In [ ]:
# Paso 1: Crear contenedor del modelo
m = Container()

# Paso 2: Definir conjuntos
ropa = Set(m, name="ropa", records=["camisa", "shorts", "pantoles"])
recurso = Set(m, name="recurso", records=["labor", "tela"])

# Paso 3: Parámetros del modelo

# Utilidad y costo fijo por producto
utilidad = Parameter(m, name="utilidad", domain=[ropa], records=np.array([6,4,7]))
costo_fijo = Parameter(m, name="costo_fijo", domain=[ropa], records=np.array([200,150,100]))

# Recurso disponible
disponibilidad = Parameter(m, name="disponibilidad", domain=[recurso], records=np.array([150,160]))

# Cant. requerida de cada recurso por cada prenda
req_recurso = Parameter(m, name="req_recurso", domain=[ropa, recurso], records=np.array([
   [3,4],
   [2,3],
   [6,4]
]))

M = 1000

# Paso 6: Variables de decision
# Cantidad a fabricar de cada tipo de ropa
x = Variable(m, name="x", type="integer", domain=[ropa])

# Variable binaria que indica si se produce o no el tipo de ropa
y = Variable(m, name="y", type="binary", domain=[ropa])

# Paso 5: Definir restricciones

# Restricción de capacidad por departamento
res_cap_req = Equation(m, name="res_cap_req", domain=[recurso])
res_cap_req[...] = Sum(ropa, req_recurso[ropa, recurso] * x[ropa]) <= disponibilidad[recurso]


# Restricción de activación de la fabricación de algún tipo de ropa
act_prod = Equation(m, name="act_prod", domain=[ropa])
act_prod[...] = x[ropa] <= M * y[ropa]

# Paso 6: Función objetivo -> maximizar la utilidad total
objetivo = Sum(ropa, utilidad[ropa] * x[ropa]) - Sum(ropa, costo_fijo[ropa] * y[ropa])

# Paso 7: Crear y resolver el modelo
modelo = Model(m,
               name="produccion",
               equations=[res_cap_req, act_prod],
               sense=Sense.MAX,
               objective=objetivo,
               problem="MIP")
modelo.solve()

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,75.0,6,7,MIP,CPLEX,0.015


In [ ]:
x.records

,ropa,level,marginal,lower,upper,scale
0,camisa,0.0,6.0,0.0,inf,1.0
1,shorts,0.0,4.0,0.0,inf,1.0
2,pantoles,25.0,7.0,0.0,inf,1.0


In [ ]:
y.records

,ropa,level,marginal,lower,upper,scale
0,camisa,0.0,-200.0,0.0,1.0,1.0
1,shorts,0.0,-150.0,0.0,1.0,1.0
2,pantoles,1.0,-100.0,0.0,1.0,1.0


In [ ]:
res_cap_req.records

,recurso,level,marginal,lower,upper,scale
0,labor,150.0,0.0,-inf,150.0,1.0
1,tela,100.0,0.0,-inf,160.0,1.0
